# Bölüm 18 — IBM SPSS Uygulamaları — Öğrenci Çalışma Defteri

Önce `VERI.md` ve `GOREVLER.md` dosyalarını okuyun. Son kontrolde `COZUMLER.md` ve `RAPORLAMA.md` kullanın. Bu defter Python referansıdır; SPSS yürütmesi değildir.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
HERE = Path.cwd()
uyku = pd.read_csv(HERE / 'uyku_esli.csv')
dis = pd.read_csv(HERE / 'dis_veri.csv')
print('Uyku:', uyku.shape, 'ToothGrowth:', dis.shape)

## 1. Eşli veri: satır sayısı ile bağımsız kişi sayısını ayırın
`uyku_esli.csv` ID üzerinden eşlenmiştir. Fark yönü koşul 2 eksi koşul 1'dir.

In [ ]:
d = uyku['fark'].to_numpy(float)
r = stats.ttest_1samp(d, 0)
print('n=', len(d), 'ortalama=', d.mean(), 's=', d.std(ddof=1))
print('t=', r.statistic, 'p=', r.pvalue, 'GA=', r.confidence_interval())
print('korelasyon=', stats.pearsonr(uyku['kosul1'], uyku['kosul2']))

## 2. Bağımsız test: aynı ham fark, farklı varyans modeli
Yalnız 1 mg/gün kayıtlarını seçin; Welch ana model, Student karşılaştırmadır. Levene p'sini otomatik model anahtarı olarak kullanmayın.

In [ ]:
doz1 = dis[np.isclose(dis['dose'], 1.0)]
oj = doz1.loc[doz1['supp']=='OJ','len'].to_numpy(float)
vc = doz1.loc[doz1['supp']=='VC','len'].to_numpy(float)
print('Welch:', stats.ttest_ind(oj, vc, equal_var=False))
print('Student:', stats.ttest_ind(oj, vc, equal_var=True))
print('Levene(mean):', stats.levene(oj, vc, center='mean'))

## 3. Durum deneyleri
İlk beş ID filtresi, ağırlık 2 ile mekanik tekrar ve iki hücre gizleme senaryolarını ana analizden ayrı değerlendirin.

In [ ]:
ilk5 = uyku.loc[uyku['ID']<=5, 'fark'].to_numpy(float)
print('İlk 5:', stats.ttest_1samp(ilk5, 0))
tekrar = np.repeat(d, 2)
print('Mekanik tekrar:', len(tekrar), stats.ttest_1samp(tekrar, 0))
kopya = uyku.copy()
kopya.loc[kopya['ID']==1, 'kosul2'] = np.nan
kopya.loc[kopya['ID']==2, 'kosul1'] = np.nan
kopya['fark'] = kopya['kosul2'] - kopya['kosul1']
tam = kopya['fark'].dropna().to_numpy(float)
print('Eksik kopya tam çift:', len(tam), stats.ttest_1samp(tam, 0))

## 4. Teslim
`SPSS-KONTROL-LISTESI.md` ile gerçek oturum kaydını denetleyin. `analiz.sps` hazırlanmış syntax'tır; ancak SPSS'te gerçekten çalıştırılıp Viewer/SPV çıktısı kaydedilirse SPSS sonucu olarak raporlanabilir.